# Model Watch 2: chat with Gemma 3 and watch it think

Ask a question, then watch the reply form one word at a time. For every word you see the choice and its runners-up, how the guess firmed up across the layers, and which concepts were active at four depths. After each reply, a one-screen replay viewer shows the whole answer with a concept timeline: click a concept and every word where it was active lights up.

To see what the replay looks like before running anything, open the preview: `docs/index.html` in this repo (the README has its web address).

Everything here is free.

| Where | Pick | Model |
|---|---|---|
| Google Colab | Runtime → Change runtime type → **T4 GPU** | Gemma 3 4B chat (default) |
| Kaggle | Settings → Accelerator → **GPU T4 x2**, Internet **on** | Gemma 3 4B, or 12B split across both GPUs |
| Any computer, no GPU | | Gemma 3 1B chat (slow but works) |

Nothing in the model is changed. This only observes.

## 1. Get the project files

- **Colab:** leave `REPO_URL` empty and upload `model-watch.zip` when asked, or paste your GitHub URL.
- **Kaggle:** add `model-watch.zip` as a Dataset (Add Input → Upload) or paste your GitHub URL. The cell finds it.

In [ ]:
import os, glob, shutil, zipfile
REPO_URL = ""  # e.g. "https://github.com/<you>/model-watch"
ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content" if os.path.exists("/content") else os.getcwd()
os.chdir(WORK)

if not os.path.exists("model-watch"):
    if REPO_URL:
        os.system(f"git clone -q {REPO_URL} model-watch")
    elif ON_KAGGLE:
        found = glob.glob("/kaggle/input/**/model-watch.zip", recursive=True) or glob.glob("/kaggle/input/**/model-watch", recursive=True)
        if not found:
            raise FileNotFoundError("Add model-watch.zip as a Kaggle Dataset (Add Input → Upload), then rerun.")
        src = found[0]
        if src.endswith(".zip"):
            zipfile.ZipFile(src).extractall(WORK)
        else:
            shutil.copytree(src, "model-watch")
    else:
        from google.colab import files
        print("Choose model-watch.zip")
        uploaded = files.upload()
        zipfile.ZipFile(next(iter(uploaded))).extractall(WORK)

os.chdir(os.path.join(WORK, "model-watch"))
os.system("pip install -q -r requirements.txt")
print("Ready in", os.getcwd())

## 2. Unlock Gemma on Hugging Face (one time)

1. Sign in at huggingface.co and accept the license on the model page you will use, for example https://huggingface.co/google/gemma-3-4b-it. The Gemma Scope 2 repos (for example https://huggingface.co/google/gemma-scope-2-4b-it) may ask too.
2. Create a read token at https://huggingface.co/settings/tokens.
3. Save it as a secret named `HF_TOKEN`: in Colab, the key icon in the left sidebar; in Kaggle, Add-ons → Secrets. If you skip this, the next cell shows a login box.

In [ ]:
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if token:
    os.environ["HF_TOKEN"] = token
    print("Using HF_TOKEN from notebook secrets.")
else:
    from huggingface_hub import notebook_login
    notebook_login()

## 3. Pick a model and load it

- `gemma-3-4b-it`: the default. Fits a free T4.
- `gemma-3-12b-it`: smarter. Only on Kaggle's **GPU T4 x2**, with `SPLIT_ACROSS_GPUS = True`.
- `gemma-3-1b-it`: smallest. Works without a GPU.
- `qwen3-1.7b`: a reasoning model. It writes its thinking before answering, and you watch both. Concepts show "pushes toward" words only. Set `MAX_WORDS = 1000` below, since thinking is long. No Hugging Face license needed.
- `qwen3-4b`: a stronger reasoning model, without concepts (the choice and the layers only).

First load downloads the model (several GB) and four concept dictionaries (about 1.3 GB for 4B). A few minutes.

In [ ]:
import torch
from model_watch import ModelWatcher, WatchConfig, step_html, save_json, export_html, trace_to_html

PRESET = "gemma-3-4b-it"
SPLIT_ACROSS_GPUS = False  # True for gemma-3-12b-it on Kaggle GPU T4 x2

n_gpu = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(n_gpu)] or "none (use gemma-3-1b-it; it will be slow)")
cfg = WatchConfig(preset=PRESET, device_map="auto" if SPLIT_ACROSS_GPUS else None)
watcher = ModelWatcher.load(cfg, hf_token=os.environ.get("HF_TOKEN"))

### 3b. Check that everything works (about a minute)

This checks the parts that can only be tested on the real model: that every layer is read correctly, that the concept dictionaries fit the model (it reports how much of each layer's activity they explain), that replies end where they should, the thinking switch for Qwen3, Neuronpedia labels, and the live viewer's server. Each line says PASS, WARN or FAIL. If anything fails, copy the output and share it.

In [ ]:
from model_watch.check import run_checks

report = run_checks(watcher)

## 4. Chat and watch

`ask("...")` sends a message, shows the reply forming live, then opens the one-screen replay. The conversation continues across calls; `new_chat()` starts over.

Settings: `MAX_WORDS` caps the reply length (counted in tokens: words or pieces of words), `DELAY` slows the live panel so you can read along (0 = full speed).

In [ ]:
import html, time
from IPython.display import display, HTML

MAX_WORDS = 250
DELAY = 0.0
conversation = []
traces = []

def new_chat():
    conversation.clear()
    print("Started a new conversation.")

def ask(message, max_words=None, delay=None, label_lookups=400):
    max_words = max_words or MAX_WORDS
    delay = DELAY if delay is None else delay
    prompt = conversation + [{"role": "user", "content": message}]
    panel = display(HTML("<i>Thinking...</i>"), display_id=True)
    reply = ""

    def show(record, steps):
        nonlocal reply
        panel.update(HTML(step_html(record, reply, watcher.feature_info, max_words)))
        if not record.get("stop"):
            reply += record["token"]
        if delay:
            time.sleep(delay)

    trace = watcher.trace(prompt, max_words, on_step=show)
    conversation[:] = prompt + [{"role": "assistant", "content": trace["reply"]}]
    watcher.add_labels(trace, max_lookups=label_lookups)
    traces.append(trace)
    n = len(traces)
    save_json(trace, f"trace-{n}.json")
    export_html(trace, f"trace-{n}.html")
    panel.update(HTML(f"<b>Reply {n}:</b> {html.escape(trace['reply'])}<br><small>Saved trace-{n}.html and trace-{n}.json</small>"))
    display(HTML(f'<iframe srcdoc="{html.escape(trace_to_html(trace))}" style="width:100%;height:860px;border:1px solid #ccc;border-radius:8px"></iframe>'))
    return trace

In [ ]:
ask("Is AI a black box?")

In [ ]:
ask("Why can't another AI just watch it and explain it to us?")

## 4b. Or watch in the full viewer (Colab, or your own machine)

This shows the one-screen replay viewer in live mode, right below the cell. Type a question at the bottom of the conversation panel, and the reply, the layer-by-layer guesses, the concepts and the timeline fill in as each word is written. You can click back to earlier words while it keeps writing; go to the latest word to follow along again. **Stop** ends a reply early, **New chat** starts over, and when a reply is done you can save it as a replay page.

- **Colab:** run the cell; the viewer appears in the cell's output. It fits one screen when the output area is at least 1000 pixels wide; narrower, its panels stack and scroll.
- **Kaggle** can't open a notebook's web server in a browser tab, so use `ask()` above there.
- Use either the live viewer or `ask()` at a time, not both at once: they share one model.

In [ ]:
from model_watch import LiveServer

PORT = 8765
if "live" in globals():
    live.stop()  # running this cell again restarts the server
live = LiveServer(watcher, max_new_tokens=MAX_WORDS, delay=DELAY).start(port=PORT)

if ON_KAGGLE:
    print("Kaggle can't show a notebook's web server. Use ask() above instead.")
else:
    try:
        from google.colab import output
        # Colab's "as_window" variant is deprecated (browsers now block it); the iframe one is supported.
        output.serve_kernel_port_as_iframe(PORT, path="/", height="860")
    except ImportError:
        print(f"Open {live.url} in your browser.")

## 5. Download the replays

Each `trace-N.html` is a single file that opens in any browser, offline.

In [ ]:
if ON_KAGGLE:
    print("Find trace-*.html under Output (right sidebar) after saving a version, or in /kaggle/working/model-watch.")
else:
    from google.colab import files
    for i in range(1, len(traces) + 1):
        files.download(f"trace-{i}.html")

## How to read it

- **Concept timeline (bottom strip).** Rows are the strongest concepts across the reply at one layer; columns are words. Darker squares mean more active. Click or tap a row name to light up every word where it was active. Click or tap a square to jump to that word and highlight that concept; the line under the strip says which concept, which word and how active. "Find a concept" searches the labels. Switch layers with the Layer buttons ("labeled" marks the layer with Neuronpedia labels): early layers track wording and grammar, middle and late layers track topics and intent.
- **Labels.** Normal-colored text is a Neuronpedia label (for the 4B model, Model Watch asks Neuronpedia at layer 17 only, and not every feature has one). Gray "pushes toward:" text is a free label computed from the model's own weights: the words that feature makes more likely. It is often clearer at late layers than early ones.
- **Settled at layer N.** The first layer from which the top guess equals the final word and never changes again.
- **Says vs. active.** Ask the model to explain its reasoning, then look at what was active while it wrote each part. When they line up, that's evidence the explanation is faithful. When they don't, that's the gap the research is about.
- **Limits.** This shows what is active, which is evidence about why, not proof. Concrete topics read clearly; abstract multi-step reasoning is the hardest case for every current tool.